# Mashtots letters session (build v1.2)

The letters study for the Western Armenian diaspora art project: a letters arm (one token per letter), Eastern-only and Western-only
letters models, and the Western ladder for the gap-closing piece. The plan and its forecasts are in `poc/LETTERS-PREREG.md`, written
before any letters run.

**Run on a G4 runtime, top to bottom.** First `MSH-0 .. MSH-9` with `MSH_PROFILE = "core"` (they rebuild the data and the tokenizers;
`MSH-7`, the token bill, can be skipped), then `LTR-1 .. LTR-8`. The code comes from `MyDrive/x3dlab/mashtots/mashtots_src_v1.2.tar.gz`
(its sha256 is pinned in `MSH-2`). Budget (Estimate): about 4 G4 hours, about 35 compute units; `LTR-4` prints the projection from
measured speed before anything long starts. Every long cell resumes from Drive.

## Core cells: data, tokenizers, freeze

In [ ]:
#@title MSH-0 · Mashtots card + GPU guard (FIRST Mashtots cell; CPU, ~5 s, 0 GB VRAM)
import types as _msh_types
MSH = globals().get("MSH") or _msh_types.SimpleNamespace()
del _msh_types
def _msh():
    import os, sys, json, shutil, subprocess, hashlib, time
    MSH.ROOT = os.environ.get("MSH_ROOT", "/content/mashtots")     # MSH_ROOT is set only by the sandbox rehearsal
    MSH.VERSION = "mashtots-poc v1.2 (shared-kernel edition)"
    os.makedirs(MSH.ROOT, exist_ok=True)
    UNREAL = ("UnrealEditor", "UnrealBuildTool", "ShaderCompileWorker", "CrashReportClient", "blender")   # plus the VRAM and utilisation tests below (they catch Chromium benches too)
    def gpu_state():
        if not shutil.which("nvidia-smi"):
            return dict(gpu=["no GPU (nvidia-smi missing)", "-", "0", "0", "0", "-"], apps=[], unreal_procs=[])
        q = lambda a: subprocess.run(["nvidia-smi", a, "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
        apps = [x.split(", ") for x in q("--query-compute-apps=pid,process_name,used_memory").splitlines() if x.strip()]
        g = q("--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu,compute_cap").split(", ")
        procs = subprocess.run(["ps", "-eo", "pid,comm,args"], capture_output=True, text=True).stdout.splitlines()
        unreal = [p.strip()[:140] for p in procs if any(u in p for u in UNREAL[:5])]
        return dict(gpu=g, apps=apps, unreal_procs=unreal)
    def gpu_guard(need_gb=2.0):
        if getattr(MSH, "LOCAL", False):
            print("sandbox dry run: GPU guard skipped (CPU only)")
            return
        s = gpu_state()
        name, drv, tot, used, util = s["gpu"][:5]
        print(f"GPU guard: {name}, driver {drv}, {used}/{tot} MiB used, util {util}%, compute apps {len(s['apps'])}, Unreal processes {len(s['unreal_procs'])}")
        for a in s["apps"]:
            print("   app:", " | ".join(a))
        for p in s["unreal_procs"]:
            print("   unreal:", p)
        busy = s["unreal_procs"] or float(used) > 2048 or float(util) > 20
        if busy and not globals().get("MSH_FORCE_GPU", False):
            raise SystemExit("GPU guard: the GPU is in use (Unreal or another job). Nothing was started. Wait for the Unreal lane's go, "
                             "then run this cell again (or set MSH_FORCE_GPU = True in MSH-1 if the Unreal lane says the GPU is free).")
        if float(tot) - float(used) < need_gb * 1024:
            raise SystemExit(f"GPU guard: less than {need_gb} GB VRAM free; nothing was started.")
        print("GPU guard: free, go." if not busy else "GPU guard: busy, but MSH_FORCE_GPU is set: going on.")
    MSH.gpu_state, MSH.gpu_guard = gpu_state, gpu_guard
    # snapshot of the kernel for the audit in MSH-14: which globals exist (by identity), sys.path and os.environ
    if not hasattr(MSH, "SNAP"):
        g = globals()
        MSH.SNAP = dict(t=time.time(), globals={k: id(v) for k, v in g.items() if not k.startswith("_")},
                        path=list(sys.path), env=hashlib.sha256(json.dumps(sorted(os.environ.items())).encode()).hexdigest())
    s = gpu_state()
    MSH.HWCARD = dict(gpu=s["gpu"], cpus=os.cpu_count(), ram_gb=round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
                      disk_free_gb=round(shutil.disk_usage(MSH.ROOT).free / 1e9, 1), python=sys.version.split()[0],
                      kernel_torch=_ver("torch"), kernel_numpy=_ver("numpy"))
    print(MSH.VERSION)
    print(json.dumps(MSH.HWCARD, indent=1))
    print("Unreal processes now:", len(s["unreal_procs"]), "| GPU compute apps now:", len(s["apps"]))
def _ver(m):
    try:
        import importlib.metadata as md
        return md.version(m)
    except Exception:
        return "missing"
_msh(); del _msh, _ver

In [ ]:
#@title MSH-1 · Configuration (form; instant). Rerun after changing a field.
MSH_PROFILE = "smoke"  #@param ["smoke", "core", "full"]
MSH_RUN_NAME = "g4a"  #@param {type:"string"}
MSH_I_APPROVE_DOWNLOADS = False  #@param {type:"boolean"}
MSH_UNITS_PER_HOUR = 8.7  #@param {type:"number"}
MSH_MAX_HOURS = 0.0  #@param {type:"number"}
MSH_ARMS_OVERRIDE = ""  #@param {type:"string"}
MSH_SIZES_OVERRIDE = ""  #@param {type:"string"}
MSH_EXTRA_SOURCES = ""  #@param {type:"string"}
MSH_ALLOW_MIRRORS = True  #@param {type:"boolean"}
MSH_COMPILE = True  #@param {type:"boolean"}
MSH_WORKERS = 0  #@param {type:"integer"}
MSH_FORCE_GPU = False  #@param {type:"boolean"}
MSH_DRIVE_BASE = "x3dlab/mashtots"  #@param {type:"string"}
def _msh():
    import os, sys, pathlib
    MSH.IN_COLAB = "google.colab" in sys.modules
    MSH.LOCAL = os.environ.get("MP_LOCAL") == "1"            # the sandbox's dry run only; never set in Colab
    MSH.PROFILE, MSH.RUN_NAME = MSH_PROFILE, MSH_RUN_NAME
    root = pathlib.Path(MSH.ROOT)
    MSH.SRC, MSH.DATA = root / "src", root / ("local" if MSH.LOCAL else MSH.PROFILE)
    MSH.RUNS, MSH.RESULTS, MSH.DEPS = MSH.DATA / "runs", MSH.DATA / "results", root / "pydeps"
    for d in (MSH.SRC, MSH.DATA, MSH.RUNS, MSH.RESULTS):
        d.mkdir(parents=True, exist_ok=True)
    MSH.EXTRA = tuple(x.strip() for x in MSH_EXTRA_SOURCES.split(",") if x.strip())
    MSH.WORKERS = MSH_WORKERS or (max(1, min(40, (os.cpu_count() or 2) - 4)) if not MSH.LOCAL else 2)
    MSH.DRIVE_DIR = f"/content/drive/MyDrive/{MSH_DRIVE_BASE}"
    MSH.DRIVE_OUT = f"{MSH.DRIVE_DIR}/runs/{MSH.RUN_NAME}-{MSH.PROFILE}" if MSH.IN_COLAB else None
    MSH.DRIVE_RUNS = f"{MSH.DRIVE_OUT}/runs" if MSH.DRIVE_OUT else None
    print(f"{MSH.VERSION} | profile {MSH.PROFILE} | run {MSH.RUN_NAME} | downloads approved: {MSH_I_APPROVE_DOWNLOADS} | workers {MSH.WORKERS} "
          f"| local data {MSH.DATA} | Drive {MSH.DRIVE_OUT} | force GPU: {MSH_FORCE_GPU}")
_msh(); del _msh

In [ ]:
#@title MSH-2 · Source code from Drive (CPU, ~5 s). Needs MyDrive/x3dlab/mashtots/mashtots_src_v1.2.tar.gz
MSH_BUNDLE_SHA256 = "8588fcfaae860351e420fd39a7e6619367c034f91adc996241f503c4d281945f"
def _msh():
    import os, sys, tarfile, hashlib, shutil, pathlib, importlib
    if MSH.IN_COLAB and not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")                       # no force_remount: if the Unreal lane mounted it, this is skipped
    bundle = pathlib.Path(os.environ.get("MSH_BUNDLE", f"{MSH.DRIVE_DIR}/mashtots_src_v1.2.tar.gz"))
    h = hashlib.sha256(bundle.read_bytes()).hexdigest()
    print("bundle", bundle, bundle.stat().st_size, "bytes, sha256", h[:16])
    if MSH_BUNDLE_SHA256.startswith("__"):
        print("(no pinned hash in this cell; the bundle's MANIFEST is still checked file by file)")
    elif h != MSH_BUNDLE_SHA256:
        raise SystemExit(f"bundle hash differs from the one in this cell ({MSH_BUNDLE_SHA256[:16]}): stop and tell Claude")
    tmp = pathlib.Path(MSH.ROOT) / "bundle_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    with tarfile.open(bundle) as t:
        t.extractall(tmp, filter="data")
    man = dict(line.split("  ")[::-1] for line in (tmp / "MANIFEST.sha256").read_text().splitlines() if line.strip())
    bad = [n for n, v in man.items() if hashlib.sha256((tmp / n).read_bytes()).hexdigest() != v]
    if bad:
        raise SystemExit(f"bundle files do not match the MANIFEST: {bad}")
    for n in man:
        dst = (MSH.DATA / n) if n.startswith("udhr/") else (MSH.SRC / n)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(tmp / n, dst)
    shutil.rmtree(tmp, ignore_errors=True)
    if str(MSH.SRC) not in sys.path:
        sys.path.append(str(MSH.SRC))                       # appended: never shadows the Unreal lane's modules
    for m in [m for m in sys.modules if m == "nb_helpers" or m.startswith("mp_")]:
        del sys.modules[m]                                  # a rerun picks up a new bundle
    H = importlib.import_module("nb_helpers")
    assert pathlib.Path(H.__file__).parent == MSH.SRC, H.__file__
    H.PY, H.DEPS = sys.executable, str(MSH.DEPS)
    env = dict(os.environ)                                  # a private copy: the kernel's os.environ is not written
    env["PYTHONPATH"] = os.pathsep.join([str(MSH.DEPS)] + [p for p in env.get("PYTHONPATH", "").split(os.pathsep) if p])
    env["HF_HOME"] = f"{MSH.ROOT}/hf"
    env["TIKTOKEN_CACHE_DIR"] = f"{MSH.ROOT}/tiktoken"
    env["MP_UDHR_DIR"] = str(MSH.DATA / "udhr")
    env["PYTHONUNBUFFERED"] = "1"
    if MSH.IN_COLAB:
        try:
            from google.colab import userdata               # a Hugging Face token in Colab's Secrets (key icon), if you added one
            tok = userdata.get("HF_TOKEN")
            if tok:
                env["HF_TOKEN"] = tok
        except Exception:
            pass
    print("HF token for gated tokenizers:", "yes (not printed)" if env.get("HF_TOKEN") else "no (mirrors used only when byte-identical)")
    H.ENV, H.SHARED, MSH.H = env, True, H
    print("wrote", len(man), "files:")
    for n in sorted(man):
        print(f"  {man[n][:12]}  {n}")
_msh(); del _msh

In [ ]:
#@title MSH-3 · Packages (private folder) and versions (CPU, ~20 s). The kernel's packages are not changed.
def _msh():
    MSH.H.install()
    MSH.H.env_report()
_msh(); del _msh

In [ ]:
#@title MSH-4 · Doctor: GPU, model code, tokenizers, network, disk, Drive (GPU ~1-2 min, ~1 GB VRAM). Go on only on ALL GOOD.
def _msh():
    import sys, subprocess
    MSH.gpu_guard(need_gb=4)
    # mp_data is imported in a subprocess, not in the kernel: at import it puts its own folder FIRST on sys.path
    code = (f"import sys; sys.path.insert(0, {str(MSH.SRC)!r}); import mp_data as D; "
            f"print(D.plan({(MSH.PROFILE if not MSH.LOCAL else 'local')!r}, {MSH.EXTRA!r})[1])")
    total = float(subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, env=MSH.H.ENV, check=True).stdout.strip())
    # disk: the pipeline writes about 7x the download (raw + pools + encoded tokens + a 2.2 GB checkpoint at 150M); full needs about 45 GB
    MSH.OK = MSH.H.doctor(MSH.SRC, MSH.DATA, MSH.DRIVE_OUT, need_gb=max(2.0, 7 * total / 1e9), compile_=MSH_COMPILE)
_msh(); del _msh

In [ ]:
#@title MSH-5 · What would be downloaded for this profile (instant; nothing is fetched here)
def _msh():
    import sys
    MSH.H.sh([sys.executable, str(MSH.SRC / "mp_data.py"), "plan", "--profile", MSH.PROFILE if not MSH.LOCAL else "local", "--extra", ",".join(MSH.EXTRA)])
    print("\nNVIDIA: no Armenian text data in the Nemotron datasets (checked 2026-10-28); Nemotron is measured through its tokenizer only.")
    print("To go on: tick MSH_I_APPROVE_DOWNLOADS in MSH-1, run MSH-1 again, then MSH-6." if not MSH_I_APPROVE_DOWNLOADS else "Downloads approved in MSH-1.")
_msh(); del _msh

In [ ]:
#@title MSH-6 · Fetch (resume-safe) and build the data: eval sets first, then cleaned, decontaminated, deduplicated pools (CPU + network)
def _msh():
    import sys, time
    t0 = time.time()
    if MSH.LOCAL:
        print("sandbox dry run: no downloads")
    else:
        assert MSH_I_APPROVE_DOWNLOADS, "tick MSH_I_APPROVE_DOWNLOADS in MSH-1 after reading the list in MSH-5"
        MSH.H.sh([sys.executable, str(MSH.SRC / "mp_data.py"), "fetch", "--profile", MSH.PROFILE, "--root", str(MSH.DATA), "--extra", ",".join(MSH.EXTRA)])
    if not (MSH.DATA / "data_stats.json").exists():
        MSH.H.sh([sys.executable, str(MSH.SRC / "mp_data.py"), "build", "--profile", MSH.PROFILE if not MSH.LOCAL else "local", "--root", str(MSH.DATA),
                  "--workers", str(MSH.WORKERS), "--extra", ",".join(MSH.EXTRA)])
    else:
        print("data already built:", MSH.DATA / "data_stats.json")
    import json
    ds = json.loads((MSH.DATA / "data_stats.json").read_text())
    print("classifier (single UD test sentences):", ds["classifier"]["accuracy"])
    print("pools MB:", {k: round(v["bytes"] / 1e6, 1) for k, v in ds["pools"].items()})
    print("dialect x spelling (MB):", {k: round(v / 1e6, 1) for k, v in ds.get("crosstab_bytes", {}).items()})
    # annotation/: the first batch for participation/annotate.html (its KEY file stays with the coordinator, as in the original notebook)
    MSH.H.checkpoint(MSH.DATA, MSH.DRIVE_OUT, ["data_stats.json", "fetch_log.json", "annotation"])
    print(f"MSH-6 wall time {time.time() - t0:.0f} s (CPU: {MSH.HWCARD['cpus']} cores, {MSH.WORKERS} workers)")
_msh(); del _msh

In [ ]:
#@title MSH-7 · The token bill (H1): eight big-lab tokenizers on the UDHR and on four treebanks (CPU, ~2-4 min)
def _msh():
    import sys
    MSH.H.sh([sys.executable, str(MSH.SRC / "mp_tok.py"), "bill", "--root", str(MSH.DATA), "--out", str(MSH.RESULTS)]
             + (["--allow-mirrors"] if MSH_ALLOW_MIRRORS else []))
    MSH.H.checkpoint(MSH.DATA, MSH.DRIVE_OUT, ["results"])
_msh(); del _msh

In [ ]:
#@title MSH-8 · Our tokenizers: suffix inventory, bpe / mashtots / two ablations, fertility, then encode the pools (CPU)
def _msh():
    import sys, time
    t0 = time.time()
    if not (MSH.DATA / "tokenizers" / "train_report.json").exists():
        MSH.H.sh([sys.executable, str(MSH.SRC / "mp_tok.py"), "train", "--root", str(MSH.DATA), "--sample-mb", "300"])
    MSH.H.sh([sys.executable, str(MSH.SRC / "mp_tok.py"), "fertility", "--root", str(MSH.DATA), "--out", str(MSH.RESULTS)])
    MSH.H.sh([sys.executable, str(MSH.SRC / "mp_tok.py"), "encode", "--root", str(MSH.DATA), "--workers", str(MSH.WORKERS)])
    MSH.H.checkpoint(MSH.DATA, MSH.DRIVE_OUT, ["tokenizers", "results", "tokens/encode_report.json"])
    print(f"MSH-8 wall time {time.time() - t0:.0f} s")
_msh(); del _msh

In [ ]:
#@title MSH-9 · FREEZE the pre-registration and the code before any model trains (instant)
def _msh():
    MSH.FROZEN = MSH.H.freeze(MSH.SRC, MSH.DATA, MSH.PROFILE, dict(build="v1.2-shared", run=MSH.RUN_NAME, hardware=MSH.HWCARD["gpu"]))
    MSH.H.checkpoint(MSH.DATA, MSH.DRIVE_OUT, ["PREREG.md", "PREREG.sha256"])
_msh(); del _msh

## Letters cells

In [ ]:
#@title LTR-1 · Letters configuration (form; instant). Needs MSH-0 .. MSH-9 first.
MSH_LTR_RUN_NAME = "ltr1"  #@param {type:"string"}
MSH_LTR_LADDER = True  #@param {type:"boolean"}
MSH_LTR_ANCHORS = True  #@param {type:"boolean"}
MSH_LTR_SIZES_OVERRIDE = ""  #@param {type:"string"}
MSH_LTR_MAX_HOURS = 0.0  #@param {type:"number"}
def _ltr():
    import pathlib
    assert getattr(MSH, "FROZEN", None) is not None or (MSH.DATA / "PREREG.sha256").exists(), "run MSH-0 .. MSH-9 first"
    MSH.LTR = pathlib.Path(MSH.DATA) / "letters"
    MSH.LTR_RUNS = MSH.LTR / "runs"
    for d in (MSH.LTR, MSH.LTR_RUNS):
        d.mkdir(parents=True, exist_ok=True)
    MSH.LTR_DRIVE = f"{MSH.DRIVE_DIR}/letters/{MSH_LTR_RUN_NAME}" if MSH.IN_COLAB else None
    MSH.LTR_DRIVE_RUNS = f"{MSH.LTR_DRIVE}/runs" if MSH.LTR_DRIVE else None
    MSH.LTR_SPECS = MSH.H.letters_grid(MSH_LTR_SIZES_OVERRIDE, ladder=MSH_LTR_LADDER, anchors=MSH_LTR_ANCHORS)
    print(f"letters study | run {MSH_LTR_RUN_NAME} | data {MSH.DATA} | letters folder {MSH.LTR} | Drive {MSH.LTR_DRIVE}")
    print(f"{len(MSH.LTR_SPECS)} runs, cheapest first:")
    for s in MSH.LTR_SPECS:
        print(f"  {s['id']:<26} {s['arm']:<8} {s['size']:>5} {s['tokens'] / 1e6:>8.0f}M tokens  {' '.join(map(str, s['args']))}")
_ltr(); del _ltr

In [ ]:
#@title LTR-2 · Letters tokens and the Western ladder (CPU, ~3-6 min). Needs the pools (MSH-6) and the bpe tokenizer (MSH-8).
def _ltr():
    import sys, time, json
    t0 = time.time()
    src, data = str(MSH.SRC), str(MSH.DATA)
    MSH.H.sh([sys.executable, f"{src}/mp_letters.py", "selftest"])
    MSH.H.sh([sys.executable, f"{src}/mp_letters.py", "vocab", "--root", data])
    MSH.H.sh([sys.executable, f"{src}/mp_tok.py", "encode", "--root", data, "--arms", "letters", "--workers", str(MSH.WORKERS)])
    if MSH_LTR_LADDER:
        MSH.H.sh([sys.executable, f"{src}/mp_letters.py", "ladder", "--root", data, "--arm", "bpe"])
    rep = json.loads((MSH.DATA / "tokens" / "encode_report.json").read_text())
    for k in ("letters/E", "letters/W", "letters/G", "bpe/W", "bytes/W"):
        if k in rep:
            print(f"  {k:<10} {rep[k]['tokens']:>14,} tokens  {rep[k]['bytes_per_token']:.2f} bytes/token")
    print(f"LTR-2 wall time {time.time() - t0:.0f} s")
    MSH.H.checkpoint(MSH.DATA, MSH.LTR_DRIVE, ["tokens/encode_report.json", "tokens/ladder_report.json", "tokenizers/letters.json"])
_ltr(); del _ltr

In [ ]:
#@title LTR-3 · FREEZE the letters plan and the code before any letters model trains (instant)
def _ltr():
    import json, shutil, hashlib
    names = ["LETTERS-PREREG.md", "mp_letters.py", "mp_train.py", "mp_tok.py", "mp_common.py", "mp_eval.py", "mp_model.py", "nb_helpers.py",
             "letters/alphabet.py", "letters/constellation.py", "letters/constellation_page.py", "letters/gap.py"]
    h = {n: hashlib.sha256((MSH.SRC / n).read_bytes()).hexdigest() for n in names}
    shutil.copy(MSH.SRC / "LETTERS-PREREG.md", MSH.DATA / "LETTERS-PREREG.md")       # the export records this file's sha256
    p = MSH.DATA / "LETTERS-PREREG.sha256"
    if p.exists():
        old = json.loads(p.read_text())
        if old["sha256"] != h:
            diff = sorted(n for n in h if old["sha256"].get(n) != h[n])
            raise SystemExit(f"the letters plan or code changed since it was frozen ({diff}); stop and tell Claude")
        print("already frozen at", old["frozen"], "and unchanged")
    else:
        import time
        p.write_text(json.dumps(dict(frozen=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()), run=MSH_LTR_RUN_NAME, sha256=h,
                                     hardware=MSH.HWCARD["gpu"], build="v1.2-letters"), indent=1))
        print("frozen:", p)
    for n, v in h.items():
        print(f"  {v[:12]}  {n}")
    MSH.H.checkpoint(MSH.DATA, MSH.LTR_DRIVE, ["LETTERS-PREREG.md", "LETTERS-PREREG.sha256"])
_ltr(); del _ltr

In [ ]:
#@title LTR-4 · What will the letters runs cost? Measured throughput per (arm, size), then the projection (GPU, ~3-5 min)
def _ltr():
    MSH.gpu_guard(need_gb=30 if MSH_COMPILE else 45)
    MSH.LTR_BENCH = MSH.H.bench(MSH.SRC, MSH.DATA, MSH.LTR_SPECS, compile_=MSH_COMPILE)
    print(f"\n{'arm':<10}{'size':>6}{'tok/s':>12}{'peak VRAM GB':>14}   (Measured on {MSH.HWCARD['gpu'][0]})")
    for (arm, size), b in sorted(MSH.LTR_BENCH.items(), key=lambda kv: (MSH.H.C.non_embedding_params(kv[0][1]), kv[0][0])):
        pk = b.get("peak_mem_gb")
        print(f"{arm:<10}{size:>6}{b['tok_per_s']:>12,.0f}{(pk if pk is not None else float('nan')):>14.1f}")
    MSH.LTR_PLAN_ROWS, MSH.LTR_PLAN_HOURS = MSH.H.project(MSH.LTR_SPECS, MSH.LTR_BENCH, MSH_UNITS_PER_HOUR)
    MSH.H.checkpoint(MSH.DATA, MSH.LTR_DRIVE, ["bench"])
_ltr(); del _ltr

In [ ]:
#@title LTR-5 · Train the letters runs, cheapest first (GPU; resume-safe, checkpoints to Drive every 15 min, finished runs skipped)
def _ltr():
    import json, time
    MSH.gpu_guard(need_gb=30 if MSH_COMPILE else 45)
    assert (MSH.DATA / "LETTERS-PREREG.sha256").exists(), "run LTR-3 (Freeze) first"
    t0, before = time.time(), sorted(p.parent.name for p in MSH.LTR_RUNS.glob("*/done.json"))
    try:
        MSH.LTR_DONE, MSH.LTR_LEFT = MSH.H.train_grid(MSH.SRC, MSH.DATA, MSH.LTR_SPECS, MSH.LTR_RUNS, MSH.LTR_DRIVE_RUNS,
                                                      max_hours=MSH_LTR_MAX_HOURS, compile_=MSH_COMPILE)
        status = "ok"
    except BaseException as exc:
        status = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        rec = dict(start=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime(t0)), seconds=round(time.time() - t0, 1), status=status,
                   finished_before=before, finished_after=sorted(p.parent.name for p in MSH.LTR_RUNS.glob("*/done.json")),
                   compile=MSH_COMPILE, max_hours=MSH_LTR_MAX_HOURS, gpu=MSH.HWCARD["gpu"][0])
        with open(MSH.LTR / "train_wall.jsonl", "a") as fh:
            fh.write(json.dumps(rec) + "\n")
        print(f"LTR-5 session: {rec['seconds'] / 3600:.2f} h, {status}; logged to letters/train_wall.jsonl")
        MSH.H.checkpoint(MSH.LTR, MSH.LTR_DRIVE, ["train_wall.jsonl"])
_ltr(); del _ltr

In [ ]:
#@title LTR-6 · Export the letters models and the ladder table (GPU or CPU, ~1-2 min)
def _ltr():
    import sys, json
    src = str(MSH.SRC)
    MSH.H.sh([sys.executable, f"{src}/mp_letters.py", "export", "--root", str(MSH.DATA), "--runs", str(MSH.LTR_RUNS),
              "--out", str(MSH.LTR / "letters_export.json")])
    MSH.H.sh([sys.executable, f"{src}/mp_letters.py", "table", "--runs", str(MSH.LTR_RUNS), "--out", str(MSH.LTR / "ladder_table.json")])
    # the anchors against core (check A1): core's 6M numbers from the frozen g4a run
    core = {"bpe-6M": 0.827, "bytes-6M": 1.077}
    for rid, ref in core.items():
        p = MSH.LTR_RUNS / f"{rid}-anchor" / "eval.json"
        if p.exists():
            v = json.loads(p.read_text()).get("bpb_EW_equal")
            print(f"  A1 {rid}-anchor: bpb_EW {v:.4f} vs core {ref} -> {'within' if abs(v - ref) <= 0.005 else 'OUTSIDE'} 0.005")
    MSH.H.checkpoint(MSH.LTR, MSH.LTR_DRIVE, ["letters_export.json", "ladder_table.json"])
_ltr(); del _ltr

In [ ]:
#@title LTR-7 · Build the two private pages: Letter Constellation and Two Shores (CPU, ~20 s), zipped for download
def _ltr():
    import sys, shutil
    src = str(MSH.SRC)
    sky, gap = MSH.LTR / "pages" / "constellation", MSH.LTR / "pages" / "two_shores"
    MSH.H.sh([sys.executable, f"{src}/letters/constellation.py", "--export", str(MSH.LTR / "letters_export.json"), "--out", str(sky)])
    if (MSH.LTR / "ladder_table.json").exists() and MSH_LTR_LADDER:
        MSH.H.sh([sys.executable, f"{src}/letters/gap.py", "--table", str(MSH.LTR / "ladder_table.json"), "--out", str(gap)])
    z = shutil.make_archive(str(MSH.LTR / f"letters_pages_{MSH_LTR_RUN_NAME}"), "zip", MSH.LTR / "pages")
    print("pages:", sky, gap, "\nzip:", z, "(private: give it to Claude, who checks it and publishes a private preview)")
    MSH.H.checkpoint(MSH.LTR, MSH.LTR_DRIVE, ["pages", f"letters_pages_{MSH_LTR_RUN_NAME}.zip"])
_ltr(); del _ltr

In [ ]:
#@title LTR-8 · Package to Drive, then audit the shared kernel (CPU, ~10 s)
def _ltr():
    import os, sys, json, time, hashlib
    wall = [json.loads(x) for x in open(MSH.LTR / "train_wall.jsonl")] if (MSH.LTR / "train_wall.jsonl").exists() else []
    info = dict(version="mashtots-poc v1.2 letters", run=MSH_LTR_RUN_NAME, finished=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
                hardware=MSH.HWCARD, runs_done=sorted(p.parent.name for p in MSH.LTR_RUNS.glob("*/done.json")),
                runs_planned=[s["id"] for s in MSH.LTR_SPECS], projected_hours=round(getattr(MSH, "LTR_PLAN_HOURS", 0.0), 2),
                train_wall_hours=round(sum(r["seconds"] for r in wall) / 3600, 3), units_per_hour=MSH_UNITS_PER_HOUR)
    (MSH.LTR / "run_info.json").write_text(json.dumps(info, indent=1))
    MSH.H.checkpoint(MSH.LTR, MSH.LTR_DRIVE, ["run_info.json"])
    print(json.dumps(info, indent=1))
    g, snap = globals(), MSH.SNAP
    changed = [k for k, i in snap["globals"].items() if k in g and id(g[k]) != i and k != "MSH" and not k.startswith("MSH_")]
    new = sorted(k for k in g if not k.startswith("_") and k not in snap["globals"] and k != "MSH" and not k.startswith("MSH_"))
    env_same = hashlib.sha256(json.dumps(sorted(os.environ.items())).encode()).hexdigest() == snap["env"]
    print("\nKERNEL AUDIT (since MSH-0): globals rebound:", changed or "none", "| new globals:", new or "none",
          "| sys.path added:", [p for p in sys.path if p not in snap["path"]] or "none", "| os.environ unchanged:", env_same)
    print("DONE. Letters results:", MSH.LTR_DRIVE or MSH.LTR)
_ltr(); del _ltr